In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd

In [4]:
citing_cited = pd.read_csv('./APS Data/aps-dataset-citations-2022.csv')

In [5]:
print(citing_cited)

                            citing_doi                          cited_doi
0        10.1103/PhysRevSeriesI.11.215         10.1103/PhysRevSeriesI.1.1
1        10.1103/PhysRevSeriesI.12.121       10.1103/PhysRevSeriesI.1.166
2          10.1103/PhysRevSeriesI.7.93       10.1103/PhysRevSeriesI.1.166
3        10.1103/PhysRevSeriesI.16.267        10.1103/PhysRevSeriesI.2.35
4         10.1103/PhysRevSeriesI.17.65       10.1103/PhysRevSeriesI.2.112
...                                ...                                ...
9833186    10.1103/PhysRevB.107.064202        10.1103/PhysRevB.106.214318
9833187    10.1103/PhysRevA.107.013525         10.1103/PhysRevX.12.041037
9833188    10.1103/PhysRevD.107.043032        10.1103/PhysRevD.106.124053
9833189    10.1103/PhysRevD.107.023524        10.1103/PhysRevD.106.124051
9833190    10.1103/PhysRevE.107.024126  10.1103/PhysRevResearch.4.L042051

[9833191 rows x 2 columns]


In [6]:
merged_table = pd.read_csv('./Created Data Tables/merged_table.csv')

In [7]:
print(merged_table)

         author_identifier                           doi    type  \
0                        0         10.1103/PhysRev.1.124  Person   
1                        3          10.1103/PhysRev.1.16  Person   
2                        7           10.1103/PhysRev.1.2  Person   
3                        9         10.1103/PhysRev.1.218  Person   
4                       12         10.1103/PhysRev.1.259  Person   
...                    ...                           ...     ...   
2528889            2571816  10.1103/RevModPhys.94.045008  Person   
2528890            2571817  10.1103/RevModPhys.94.045008  Person   
2528891            2571818  10.1103/RevModPhys.94.045008  Person   
2528892            2571819  10.1103/RevModPhys.94.045008  Person   
2528893            2571820  10.1103/RevModPhys.94.045008  Person   

                             name       firstname        surname  year  \
0               Lachlan Gilchrist         Lachlan      Gilchrist  1913   
1             David W. Cornelius.  

### Graph Neural Network

In [8]:
import numpy as np
np.random.seed(22)

In [9]:
import torch
from torch import Tensor
import torch.nn.functional as F
import torch_geometric.transforms as T
from sklearn.metrics import roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.loader import LinkNeighborLoader
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report

##### Heterogeneous Graph Creation

In [10]:
# subset merged_table and citing_cited for "PhysRevE"
sampled_merged_table_physreve = merged_table[merged_table["doi"].str.contains("PhysRevE")]
sampled_citing_cited_physreve = citing_cited[citing_cited['citing_doi'].str.contains("PhysRevE")]

In [11]:
in_channels = 7

In [12]:
# mapping IDs
sampled_merged_table_physreve['author_identifier'] = sampled_merged_table_physreve['author_identifier'].astype('category')
paper_id_mapping = {doi: idx for idx, doi in enumerate(pd.concat([sampled_citing_cited_physreve['citing_doi'], sampled_citing_cited_physreve['cited_doi']]).unique())}
author_id_mapping = {author_id: idx for idx, author_id in enumerate(sampled_merged_table_physreve['author_identifier'].cat.categories)}

# assign numeric codes to the 'author_identifier' column
sampled_merged_table_physreve['author_id_code'] = sampled_merged_table_physreve['author_identifier'].cat.codes

In [13]:
countries_df = pd.DataFrame(merged_table['countries'])
countries_df = countries_df[countries_df.index.isin(author_id_mapping.keys())]

# ensure all entries in 'countries' are lists and handle any empty entries
def ensure_list(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, str):
        try:
            return eval(x)
        except:
            return []
    else:
        return []

countries_list = countries_df['countries'].apply(ensure_list)

# flatten the list of countries to get a list of unique countries
countries_list_flat = [country for sublist in countries_list for country in sublist]
unique_countries = list(set(countries_list_flat))

# one-hot encode the 'countries' column
encoder = OneHotEncoder(categories=[unique_countries], sparse=False, handle_unknown='ignore')

# transform the list of countries into a single string for each entry
countries_str_list = countries_list.apply(lambda x: ','.join(sorted(set(x))) if x else 'none')

# fit the encoder
encoded_countries = encoder.fit_transform(countries_str_list.values.reshape(-1, 1))

# convert the dense array to a tensor and add it as a feature to authors
author_country_features = torch.tensor(encoded_countries, dtype=torch.float)

# create a dictionary mapping DOIs to years
doi_to_year = dict(zip(sampled_merged_table_physreve['paper_identifier'], sampled_merged_table_physreve['year']))

# create a list of years corresponding to the `paper_id_mapping`, filling with a default value if not found
years = [doi_to_year.get(doi, 0) for doi in paper_id_mapping.keys()]

# normalize the 'year' column
scaler = MinMaxScaler()
normalized_years = scaler.fit_transform(pd.Series(years).values.reshape(-1, 1))

# convert to a tensor and add as a feature to papers
paper_year_features = torch.tensor(normalized_years, dtype=torch.float)

# define vector of ones
num_paper_nodes = len(paper_id_mapping)
num_author_nodes = len(author_id_mapping)

# define the feature dimensions
num_year_features = normalized_years.shape[1]  
num_country_features = encoded_countries.shape[1]  

# define arrays that match the shape of your vector of ones
paper_ones = torch.ones((num_paper_nodes, in_channels))  
author_ones = torch.ones((num_author_nodes, in_channels)) 

# combine with additional features
paper_features_shape = (num_paper_nodes, in_channels + num_year_features)  
author_features_shape = (num_author_nodes, in_channels + num_country_features)  

In [14]:
# initialize edge indices
edge_index_citation = []
edge_index_authorship = []

# populate edge_index_citation
for _, row in sampled_citing_cited_physreve.iterrows():
    citing_doi = row['citing_doi']
    cited_doi = row['cited_doi']
    
    if citing_doi in paper_id_mapping and cited_doi in paper_id_mapping:
        citing_id = paper_id_mapping[citing_doi]
        cited_id = paper_id_mapping[cited_doi]
        edge_index_citation.append([citing_id, cited_id])

# populate edge_index_authorship
missing_papers = 0
missing_authors = 0
for _, row in sampled_merged_table_physreve.iterrows():
    doi = row['doi']
    author_identifier = row['author_identifier']
    
    if doi in paper_id_mapping and author_identifier in author_id_mapping:
        paper_id = paper_id_mapping[doi]
        author_id = author_id_mapping[author_identifier]
        edge_index_authorship.append([paper_id, author_id])
    else:
        if doi not in paper_id_mapping:
            missing_papers += 1
        if author_identifier not in author_id_mapping:
            missing_authors += 1

print()
print(f'Total missing papers: {missing_papers}')
print(f'Total missing authors: {missing_authors}')

# convert lists to torch tensors and ensure they are in contiguous memory
edge_index_citation = torch.tensor(edge_index_citation, dtype=torch.long).t().contiguous()
edge_index_authorship = torch.tensor(edge_index_authorship, dtype=torch.long).t().contiguous()


Total missing papers: 422
Total missing authors: 0


In [15]:
# create HeteroData object
data = HeteroData()

# add paper nodes and features (including the normalized year feature)
data['paper'].num_nodes = len(paper_id_mapping)
paper_features = torch.cat([torch.ones((len(paper_id_mapping), in_channels)), paper_year_features], dim=1)
data['paper'].x = paper_features

# dd author nodes and features (including the one-hot encoded country features)
data['author'].num_nodes = len(author_id_mapping)
author_features = torch.cat([torch.ones((len(author_id_mapping), in_channels)), author_country_features], dim=1)
data['author'].x = author_features

# add citation and authorship edges
data['paper', 'cites', 'paper'].edge_index = edge_index_citation
data['paper', 'written_by', 'author'].edge_index = edge_index_authorship
data['author', 'writes', 'paper'].edge_index = edge_index_authorship.flip(0)

# print node counts
print('Number of \'paper\' Nodes:', data['paper'].num_nodes)
print('Number of \'author\' Nodes:', data['author'].num_nodes)
print()

# print feature dimensions
print(f'Paper Feature Shape: {paper_features_shape}')
print(f'Author Feature Shape: {author_features_shape}')
print()

# print edge index shapes
print('Citation Edge Index Shape:', tuple(data['paper', 'cites', 'paper'].edge_index.size()))
print('Authorship Edge Index Shape:', tuple(data['paper', 'written_by', 'author'].edge_index.size()))
print('Reverse Authorship Edge Index Shape:', tuple(data['author', 'writes', 'paper'].edge_index.size()))

# check node types
assert set(data.node_types) == {'paper', 'author'}, f'Unexpected node types: {data.node_types}'

# check edge types
assert set(data.edge_types) == {('paper', 'cites', 'paper'), ('paper', 'written_by', 'author'), ('author', 'writes', 'paper')}, f'Unexpected edge types: {data.edge_types}'

# check number of nodes
assert data['paper'].num_nodes == 139966, f'Unexpected number of \'paper\' nodes: {data["paper"].num_nodes}'
assert data['author'].num_nodes == 192814, f'Unexpected number of \'author\' nodes: {data["author"].num_nodes}'

# ensure the new features have been added correctly
assert data['paper'].x.shape[1] == in_channels + 1, f'Unexpected number of \'paper\' features: {data["paper"].x.shape[1]}'
assert data['author'].x.shape[1] == in_channels + len(unique_countries), f'Unexpected number of \'author\' features: {data["author"].x.shape[1]}'

# check number of edges
assert data['paper', 'cites', 'paper'].edge_index.shape[1] == 700250, f'Unexpected number of \'cites\' edges: {data["paper", "cites", "paper"].edge_index.shape[1]}'
assert data['paper', 'written_by', 'author'].edge_index.shape[1] == 192392, f'Unexpected number of \'written_by\' edges: {data["paper", "written_by", "author"].edge_index.shape[1]}'
assert data['author', 'writes', 'paper'].edge_index.shape[1] == 192392, f'Unexpected number of \'writes\' edges: {data["author", "writes", "paper"].edge_index.shape[1]}'

print('\nAll assertions passed!')

Number of 'paper' Nodes: 139966
Number of 'author' Nodes: 192814

Paper Feature Shape: (139966, 8)
Author Feature Shape: (192814, 22)

Citation Edge Index Shape: (2, 700250)
Authorship Edge Index Shape: (2, 192392)
Reverse Authorship Edge Index Shape: (2, 192392)

All assertions passed!


##### Edge-Level Splits

In [16]:
# transform for splitting the dataset
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=False,
    edge_types=('paper', 'written_by', 'author'),
    rev_edge_types=('author', 'writes', 'paper'),
)

# apply the transform to split the data
train_data, val_data, test_data = transform(data)

# print split data details
print('Training data:')
print('==============')
print(train_data)
print()
print('Validation data:')
print('================')
print(val_data)
print()
print('Testing data:')
print('=============')
print(test_data)
print()

assert train_data['paper', 'written_by', 'author'].num_edges == 107740, \
    f"Unexpected number of 'written_by' edges in training data: {train_data['paper', 'written_by', 'author'].num_edges}"
assert train_data['paper', 'written_by', 'author'].edge_label_index.size(1) == 46174, \
    f"Unexpected edge_label_index size in training data: {train_data['paper', 'written_by', 'author'].edge_label_index.size(1)}"
assert train_data['author', 'writes', 'paper'].num_edges == 107740, \
    f"Unexpected number of 'writes' edges in training data: {train_data['author', 'writes', 'paper'].num_edges}"

# no negative edges added
assert train_data['paper', 'written_by', 'author'].edge_label.min() == 1, \
    f"Unexpected min edge_label in training data: {train_data['paper', 'written_by', 'author'].edge_label.min()}"
assert train_data['paper', 'written_by', 'author'].edge_label.max() == 1, \
    f"Unexpected max edge_label in training data: {train_data['paper', 'written_by', 'author'].edge_label.max()}"

assert val_data['paper', 'written_by', 'author'].num_edges == 153914, \
    f"Unexpected number of 'written_by' edges in validation data: {val_data['paper', 'written_by', 'author'].num_edges}"
assert val_data['paper', 'written_by', 'author'].edge_label_index.size(1) == 57717, \
    f"Unexpected edge_label_index size in validation data: {val_data['paper', 'written_by', 'author'].edge_label_index.size(1)}"
assert val_data['author', 'writes', 'paper'].num_edges == 153914, \
    f"Unexpected number of 'writes' edges in validation data: {val_data['author', 'writes', 'paper'].num_edges}"

# negative edges with ratio 2:1
assert val_data['paper', 'written_by', 'author'].edge_label.long().bincount().tolist() == [38478, 19239], \
    f"Unexpected edge_label bincount in validation data: {val_data['paper', 'written_by', 'author'].edge_label.long().bincount().tolist()}"

assert test_data['paper', 'written_by', 'author'].num_edges == 173153, \
    f"Unexpected number of 'written_by' edges in testing data: {test_data['paper', 'written_by', 'author'].num_edges}"
assert test_data['paper', 'written_by', 'author'].edge_label_index.size(1) == 57717, \
    f"Unexpected edge_label_index size in testing data: {test_data['paper', 'written_by', 'author'].edge_label_index.size(1)}"
assert test_data['author', 'writes', 'paper'].num_edges == 173153, \
    f"Unexpected number of 'writes' edges in testing data: {test_data['author', 'writes', 'paper'].num_edges}"

print('\nAll assertions passed!')

Training data:
HeteroData(
  paper={
    num_nodes=139966,
    x=[139966, 8],
  },
  author={
    num_nodes=192814,
    x=[192814, 22],
  },
  (paper, cites, paper)={ edge_index=[2, 700250] },
  (paper, written_by, author)={
    edge_index=[2, 107740],
    edge_label=[46174],
    edge_label_index=[2, 46174],
  },
  (author, writes, paper)={ edge_index=[2, 107740] }
)

Validation data:
HeteroData(
  paper={
    num_nodes=139966,
    x=[139966, 8],
  },
  author={
    num_nodes=192814,
    x=[192814, 22],
  },
  (paper, cites, paper)={ edge_index=[2, 700250] },
  (paper, written_by, author)={
    edge_index=[2, 153914],
    edge_label=[57717],
    edge_label_index=[2, 57717],
  },
  (author, writes, paper)={ edge_index=[2, 153914] }
)

Testing data:
HeteroData(
  paper={
    num_nodes=139966,
    x=[139966, 8],
  },
  author={
    num_nodes=192814,
    x=[192814, 22],
  },
  (paper, cites, paper)={ edge_index=[2, 700250] },
  (paper, written_by, author)={
    edge_index=[2, 173153],
    

##### Mini-Batch Loaders

In [17]:
# define seed edges
edge_label_index = train_data['paper', 'written_by', 'author'].edge_label_index
edge_label = train_data['paper', 'written_by', 'author'].edge_label

# create a LinkNeighborLoader
train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[20, 10],
    neg_sampling_ratio=2.0,
    edge_label_index=(('paper', 'written_by', 'author'), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
)

# inspect a sample
sampled_data = next(iter(train_loader))

print('Sampled mini-batch:')
print('===================')
print(sampled_data)

# perform assertions
assert sampled_data['paper', 'written_by', 'author'].edge_label_index.size(1) == 3 * 128, \
    f"Unexpected edge_label_index size in sampled data: {sampled_data['paper', 'written_by', 'author'].edge_label_index.size(1)}"
assert sampled_data['paper', 'written_by', 'author'].edge_label.min() == 0, \
    f"Unexpected min edge_label in sampled data: {sampled_data['paper', 'written_by', 'author'].edge_label.min()}"
assert sampled_data['paper', 'written_by', 'author'].edge_label.max() == 1, \
    f"Unexpected max edge_label in sampled data: {sampled_data['paper', 'written_by', 'author'].edge_label.max()}"

print('\nAll assertions passed!')

Sampled mini-batch:
HeteroData(
  paper={
    num_nodes=5284,
    x=[5284, 8],
    n_id=[5284],
  },
  author={
    num_nodes=3500,
    x=[3500, 22],
    n_id=[3500],
  },
  (paper, cites, paper)={
    edge_index=[2, 6511],
    e_id=[6511],
  },
  (paper, written_by, author)={
    edge_index=[2, 600],
    edge_label=[384],
    edge_label_index=[2, 384],
    e_id=[600],
    input_id=[128],
  },
  (author, writes, paper)={
    edge_index=[2, 3257],
    e_id=[3257],
  }
)

All assertions passed!


##### Heterogeneous Link-Level GNN

In [18]:
# define the GNN model
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # define four SAGEConv layers
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='sum')
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='sum')
        self.conv3 = SAGEConv(hidden_channels, hidden_channels, aggr='sum')
        self.conv4 = SAGEConv(hidden_channels, hidden_channels, aggr='sum')

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        # apply the SAGEConv layers with ReLU activation
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        # apply the SAGEConv layer without activation
        x = self.conv4(x, edge_index)
        return x

# define the classifier
class Classifier(torch.nn.Module):
    def forward(self, x_paper: Tensor, x_author: Tensor, edge_label_index: Tensor) -> Tensor:
        # get the paper and author features at the specified edges
        edge_feat_paper = x_paper[edge_label_index[0]]
        edge_feat_author = x_author[edge_label_index[1]]
        # compute the dot product and sum it up
        return (edge_feat_paper * edge_feat_author).sum(dim=-1)

# define the overall model
class Model(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_paper_features, num_author_features):
        super().__init__()
        # linear transformation for paper features
        self.paper_lin = torch.nn.Linear(num_paper_features, in_channels)
        # linear transformation for author features
        self.author_lin = torch.nn.Linear(num_author_features, in_channels)

        # instantiate homogeneous GNN with correct input dimensions
        self.gnn = GNN(in_channels, hidden_channels)

        # convert GNN model into a heterogeneous variant
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())

        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        # create a dictionary of transformed features
        x_dict = {
            'paper': self.paper_lin(data['paper'].x),
            'author': self.author_lin(data['author'].x),
        }

        # apply the GNN to the features and edge indices
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        
        # classify the edges based on the node features
        pred = self.classifier(
            x_dict['paper'],
            x_dict['author'],
            data['paper', 'written_by', 'author'].edge_label_index,
        )

        return pred

num_paper_features = in_channels + 1  # account for the normalized year feature
num_author_features = in_channels + len(unique_countries)  # account for the one-hot encoded country features
hidden_channels = 64

model = Model(in_channels, hidden_channels, num_paper_features, num_author_features)

print(model)

Model(
  (paper_lin): Linear(in_features=8, out_features=7, bias=True)
  (author_lin): Linear(in_features=22, out_features=7, bias=True)
  (gnn): GraphModule(
    (conv1): ModuleDict(
      (paper__cites__paper): SAGEConv(7, 64, aggr=sum)
      (paper__written_by__author): SAGEConv(7, 64, aggr=sum)
      (author__writes__paper): SAGEConv(7, 64, aggr=sum)
    )
    (conv2): ModuleDict(
      (paper__cites__paper): SAGEConv(64, 64, aggr=sum)
      (paper__written_by__author): SAGEConv(64, 64, aggr=sum)
      (author__writes__paper): SAGEConv(64, 64, aggr=sum)
    )
    (conv3): ModuleDict(
      (paper__cites__paper): SAGEConv(64, 64, aggr=sum)
      (paper__written_by__author): SAGEConv(64, 64, aggr=sum)
      (author__writes__paper): SAGEConv(64, 64, aggr=sum)
    )
    (conv4): ModuleDict(
      (paper__cites__paper): SAGEConv(64, 64, aggr=sum)
      (paper__written_by__author): SAGEConv(64, 64, aggr=sum)
      (author__writes__paper): SAGEConv(64, 64, aggr=sum)
    )
  )
  (classifie

##### Per-Epoch Activity 

In [19]:
# define the validation seed edges
edge_label_index = val_data['paper', 'written_by', 'author'].edge_label_index  
edge_label = val_data['paper', 'written_by', 'author'].edge_label  

# create a LinkNeighborLoader for validation data
val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[20, 10], 
    edge_label_index=(('paper', 'written_by', 'author'), edge_label_index),  
    edge_label=edge_label,  
    batch_size=3 * 128,  
    shuffle=False,  
)

# sample a mini-batch from validation loader
sampled_data = next(iter(val_loader))

print('Sampled mini-batch:')
print('===================')
print(sampled_data)

# perform assertions
assert sampled_data['paper', 'written_by', 'author'].edge_label_index.size(1) == 3 * 128  
assert sampled_data['paper', 'written_by', 'author'].edge_label.min() >= 0  
assert sampled_data['paper', 'written_by', 'author'].edge_label.max() <= 1  

print('\nAll assertions passed!')  

Sampled mini-batch:
HeteroData(
  paper={
    num_nodes=4368,
    x=[4368, 8],
    n_id=[4368],
  },
  author={
    num_nodes=4603,
    x=[4603, 22],
    n_id=[4603],
  },
  (paper, cites, paper)={
    edge_index=[2, 5425],
    e_id=[5425],
  },
  (paper, written_by, author)={
    edge_index=[2, 1000],
    edge_label=[384],
    edge_label_index=[2, 384],
    e_id=[1000],
    input_id=[384],
  },
  (author, writes, paper)={
    edge_index=[2, 4219],
    e_id=[4219],
  }
)

All assertions passed!


##### Training 

In [20]:
from tqdm import tqdm

torch.manual_seed(22)

device = torch.device('cpu')  # set device to CPU explicitly
print(f'Device: {device}')

# move the model to the specified device
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(model, data_loader, device):
    model.eval()
    total_loss = total_examples = 0
    with torch.no_grad():
        for sampled_data in data_loader:

            # move data to the device
            sampled_data = sampled_data.to(device)

            # get model predictions
            pred = model(sampled_data)

            # ground truth labels are in the edge_label for "written_by" relation
            ground_truth = sampled_data['paper', 'written_by', 'author'].edge_label.to(device)

            # compute loss
            loss = F.binary_cross_entropy_with_logits(pred, ground_truth.float())

            # accumulate the loss and the number of examples
            total_loss += float(loss) * pred.numel()
            total_examples += pred.numel()
    
    # calculate and print the average validation loss
    return total_loss / total_examples


for epoch in range(1, 5):
    model.train()
    total_loss = total_examples = 0
    for sampled_data in tqdm(train_loader):
        optimizer.zero_grad()

        # move data to the device 
        sampled_data = sampled_data.to(device)
        
        # get model predictions
        pred = model(sampled_data)

        # ground truth labels are in the edge_label for "written_by" relation
        ground_truth = sampled_data['paper', 'written_by', 'author'].edge_label.to(device)
        
        # compute loss
        loss = F.binary_cross_entropy_with_logits(pred, ground_truth.float())
        
        # backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # accumulate the loss and the number of examples
        total_loss += float(loss) * pred.numel()
        total_examples += pred.numel()
    
    # calculate and print the average training loss
    train_loss = total_loss / total_examples
    print(f'Epoch: {epoch:03d}, Training Loss: {train_loss:.4f}')
    
    # evaluate on the validation data
    val_loss = evaluate(model, val_loader, device)
    print(f'Epoch: {epoch:03d}, Validation Loss: {val_loss:.4f}')

Device: cpu


100%|██████████| 361/361 [00:42<00:00,  8.52it/s]


Epoch: 001, Training Loss: 0.3835
Epoch: 001, Validation Loss: 0.2305


100%|██████████| 361/361 [00:38<00:00,  9.36it/s]


Epoch: 002, Training Loss: 0.3633
Epoch: 002, Validation Loss: 0.2266


100%|██████████| 361/361 [00:42<00:00,  8.58it/s]


Epoch: 003, Training Loss: 0.3617
Epoch: 003, Validation Loss: 0.2253


100%|██████████| 361/361 [00:39<00:00,  9.03it/s]


Epoch: 004, Training Loss: 0.3624
Epoch: 004, Validation Loss: 0.2236


In [21]:
total_params = 0

# iterate over all named parameters in the model
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        print("Non-Trainable Parameters:", name)  
    params = parameter.numel() 
    print([name, parameter.size(), params])  
    total_params += params  

print()
print(f"Total Trainable Paramters: {total_params}")  

['paper_lin.weight', torch.Size([7, 8]), 56]
['paper_lin.bias', torch.Size([7]), 7]
['author_lin.weight', torch.Size([7, 22]), 154]
['author_lin.bias', torch.Size([7]), 7]
['gnn.conv1.paper__cites__paper.lin_l.weight', torch.Size([64, 7]), 448]
['gnn.conv1.paper__cites__paper.lin_l.bias', torch.Size([64]), 64]
['gnn.conv1.paper__cites__paper.lin_r.weight', torch.Size([64, 7]), 448]
['gnn.conv1.paper__written_by__author.lin_l.weight', torch.Size([64, 7]), 448]
['gnn.conv1.paper__written_by__author.lin_l.bias', torch.Size([64]), 64]
['gnn.conv1.paper__written_by__author.lin_r.weight', torch.Size([64, 7]), 448]
['gnn.conv1.author__writes__paper.lin_l.weight', torch.Size([64, 7]), 448]
['gnn.conv1.author__writes__paper.lin_l.bias', torch.Size([64]), 64]
['gnn.conv1.author__writes__paper.lin_r.weight', torch.Size([64, 7]), 448]
['gnn.conv2.paper__cites__paper.lin_l.weight', torch.Size([64, 64]), 4096]
['gnn.conv2.paper__cites__paper.lin_l.bias', torch.Size([64]), 64]
['gnn.conv2.paper__cite

##### Evaluation

In [22]:
# define the test seed edges
edge_label_index_test = test_data['paper', 'written_by', 'author'].edge_label_index  
edge_label_test = test_data['paper', 'written_by', 'author'].edge_label  

# create a LinkNeighborLoader for test data
test_loader = LinkNeighborLoader(
    data=test_data,
    num_neighbors=[20, 10], 
    edge_label_index=(('paper', 'written_by', 'author'), edge_label_index_test),  
    edge_label=edge_label_test,  
    batch_size=3 * 128,  
    shuffle=False,  
)

In [24]:
import tqdm

# evaluate on the test data
test_loss = evaluate(model, test_loader, device)
print(f'Test Loss: {test_loss:.4f}')

# get predictions and ground truths for test set
preds_test = []  
ground_truths_test = []  

# iterate over test loader, using tqdm for progress bar
for sampled_data in tqdm.tqdm(test_loader):
    with torch.no_grad():
        sampled_data.to(device)  
        preds_test.append(model(sampled_data)) 
        ground_truths_test.append(sampled_data['paper', 'written_by', 'author'].edge_label)  

# concatenate predictions and ground truths
pred_test = torch.cat(preds_test, dim=0).cpu().numpy()  
ground_truth_test = torch.cat(ground_truths_test, dim=0).cpu().numpy()  

# compute and print test AUC score
test_auc = roc_auc_score(ground_truth_test, pred_test)  
print()
print(f'Test AUC: {test_auc:.4f}') 

# convert predictions to binary labels based on threshold 
pred_labels_test = (torch.sigmoid(torch.tensor(pred_test)) > 0.5).numpy()

# compute confusion matrix
conf_matrix_test = confusion_matrix(ground_truth_test, pred_labels_test)
print(f"\nConfusion Matrix (Test):\n{conf_matrix_test}")

# print classification report
print("\nClassification Report (Test):")
print(classification_report(ground_truth_test, pred_labels_test, target_names=['Negative', 'Positive']))

Test Loss: 0.1635


100%|██████████| 151/151 [00:06<00:00, 23.16it/s]



Test AUC: 0.9771

Confusion Matrix (Test):
[[36628  1850]
 [  869 18370]]

Classification Report (Test):
              precision    recall  f1-score   support

    Negative       0.98      0.95      0.96     38478
    Positive       0.91      0.95      0.93     19239

    accuracy                           0.95     57717
   macro avg       0.94      0.95      0.95     57717
weighted avg       0.95      0.95      0.95     57717

